# Question 4 - Testing Tool Calls

Question 3 tested: **Did the final answer contain the correct result?**

Question 4 tests: **Did the agent follow the intended process?**

```text
Question 3
RESULT TEST
    ↓
Did the output contain the correct answer?


Question 4
BEHAVIOR / TRAJECTORY TEST
    ↓
Did the agent call:

get_schema
    ↓
run_sql
```

We inspect the actual message history rather than infer behavior from the agent's instructions. `collect_tools` extracts `ToolCallPart` objects in message order, parses their arguments, and excludes PydanticAI's `final_result` structured-output tool. Run from `week3_homework/` with an `OPENAI_API_KEY` available.

In [1]:
from sql_agent import agent
from utils import collect_tools

user_prompt = "What is the most common payment type?"
result = await agent.run(user_prompt)
messages = result.new_messages()
tool_calls = collect_tools(messages)

                 pydantic-ai v2.63.0 • Python 3.13.15

      / \        agent: agent • model: openai:gpt-4o-mini • output: SQLResult • tools: 2
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


In [2]:
for index, call in enumerate(tool_calls, start=1):
    print(f"{index}. {call.name}({call.args})")

1. get_schema({})
2. run_sql({'query': 'SELECT payment_type, COUNT(*) AS count\nFROM trips\nGROUP BY payment_type\nORDER BY count DESC\nLIMIT 1;'})


In [3]:
assert len(tool_calls) >= 2
assert tool_calls[0].name == "get_schema"
assert any(call.name == "run_sql" for call in tool_calls)

second_tool = tool_calls[1].name
print("Second tool:", second_tool)
print("PASS: schema inspection is first and SQL execution is present.")

Second tool: run_sql
PASS: schema inspection is first and SQL execution is present.


The collected calls show the order in which the model requested application tools. They do not include tool-return messages or the structured final-output tool. The test does not require one exact SQL string, since equivalent SQL queries can answer the question.

Run both homework tests with:

```bash
uv run pytest test_agent.py -s
```

**Name of the second tool the agent calls: `run_sql`.**

This was observed in the actual completed agent run above, and the pytest run also observed `get_schema` followed by `run_sql`. Both Question 3 and Question 4 tests passed (`2 passed`).